# Project 4: Linear Regression from Scratch**Date**: 2026-09-12  **Estimated Time**: 15-20 hours  **Goal**: Implement linear regression from scratch and understand how it works

## Learning ObjectivesBy the end of this project, you will:- Understand linear regression mathematically- Derive the cost function (MSE)- Implement gradient descent from scratch- Understand learning rate and convergence- Implement regularization (Ridge, Lasso, Elastic Net)- Evaluate regression models- Perform residual analysis- Compare with scikit-learn

In [ ]:
# Setup: import librariesimport numpy as npimport pandas as pdimport matplotlib.pyplot as plt%matplotlib inlinenp.random.seed(42)print('Setup complete!')print(f'NumPy version: {np.__version__}')

## 1. Problem Definition**Problem**: Predict house prices using linear regression.**Type**: Supervised learning - regression**Goal**: Given features of a house (size, bedrooms, age, etc.), predict its price.**Why this matters**: Linear regression is the simplest and most interpretable ML algorithm. Understanding it deeply is essential for everything that follows.**What makes this challenging**:- It is your first real ML algorithm- You need both calculus and linear algebra- Gradient descent can be slow or fail to converge- Features must be scaled for some methods

## 2. Mathematical Background### Hypothesis FunctionFor one feature: h(x) = w0 + w1 * xFor many features: h(x) = w0 + w1 * x1 + ... + wn * xnIn vector form with an added column of ones for w0:h(X) = X @ w### Cost Function (Mean Squared Error)J(w) = (1 / 2m) * sum((h(X) - y) ** 2)We divide by 2 to make the derivative cleaner.### Gradient Descent Updatew = w - alpha * (1 / m) * X.T @ (X @ w - y)where alpha is the learning rate.### Normal Equation (closed form)w = (X.T @ X)^-1 @ X.T @ y

## 3. Dataset CreationWe create a synthetic house price dataset. The price depends on size, bedrooms, age, and distance to the city.

In [ ]:
# Generate synthetic house datan = 150# Featuressize = np.random.normal(1500, 400, n)bedrooms = np.random.randint(1, 6, n)age = np.random.randint(1, 50, n)distance = np.random.normal(15, 8, n)# Price with known coefficients plus noiseprice = (50 * size +         10000 * bedrooms -         500 * age -         1000 * distance +         50000 +         np.random.normal(0, 15000, n))df = pd.DataFrame({    'size': size.round(2),    'bedrooms': bedrooms,    'age': age,    'distance': distance.round(2),    'price': price.round(2)})# Manual train/test split (80 / 20)indices = np.random.permutation(n)split = int(0.8 * n)train_idx = indices[:split]test_idx = indices[split:]train = df.iloc[train_idx].copy()test = df.iloc[test_idx].copy()print(f'Train size: {len(train)}')print(f'Test size:  {len(test)}')print('\nFirst 5 rows:')print(df.head())

## 4. Simple Linear Regression from ScratchStart with one feature (size) to predict price. This makes the mechanics clear.

In [ ]:
# Prepare data for simple regressionX_train = train[['size']].valuesy_train = train['price'].valuesX_test = test[['size']].valuesy_test = test['price'].values# Add intercept (column of ones)X_train_b = np.hstack([np.ones((X_train.shape[0], 1)), X_train])X_test_b = np.hstack([np.ones((X_test.shape[0], 1)), X_test])# Normal equation for simple regressionw = np.linalg.pinv(X_train_b.T @ X_train_b) @ X_train_b.T @ y_trainprint(f'Intercept w0: {w[0]:.2f}')print(f'Slope w1:     {w[1]:.2f}')# Predicty_pred = X_test_b @ w# Mean Squared Errormse = np.mean((y_test - y_pred) ** 2)rmse = np.sqrt(mse)mae = np.mean(np.abs(y_test - y_pred))print(f'\nMSE:  {mse:.2f}')print(f'RMSE: {rmse:.2f}')print(f'MAE:  {mae:.2f}')# Plotplt.figure(figsize=(8, 5))plt.scatter(X_test, y_test, alpha=0.6, label='Actual')plt.plot(X_test, y_pred, color='red', label='Predicted')plt.xlabel('Size (sq ft)')plt.ylabel('Price')plt.title('Simple Linear Regression: Size vs Price')plt.legend()plt.show()

## 5. Cost Function and Gradient DescentNow we implement gradient descent to find the weights. Watch how the cost decreases over iterations.

In [ ]:
def compute_cost(X, y, w):
    """Compute MSE cost."""
    m = len(y)
    predictions = X @ w
    return np.sum((predictions - y) ** 2) / (2 * m)


def gradient_descent(X, y, w, alpha, n_iter):
    """Train weights using batch gradient descent."""
    m = len(y)
    cost_history = []
    for i in range(n_iter):
        predictions = X @ w
        error = predictions - y
        gradient = (X.T @ error) / m
        w = w - alpha * gradient
        cost_history.append(compute_cost(X, y, w))
    return w, cost_history


# Initialize weights

w_init = np.zeros(X_train_b.shape[1])
alpha = 0.0000001
n_iter = 500
w_gd, costs = gradient_descent(X_train_b, y_train, w_init, alpha, n_iter)
print(f"GD weights: {w_gd}")
print(f"Final cost: {costs[-1]:.2f}")
plt.figure(figsize=(8, 4))
plt.plot(costs)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.title("Cost vs. Iteration")
plt.show()


## 6. Feature ScalingGradient descent works best when all features are on the same scale. We standardize to mean=0 and std=1.

In [ ]:
# Standardize featuresdef standardize(X, mean, std):    return (X - mean) / stdX_train_all = train[['size', 'bedrooms', 'age', 'distance']].valuesy_train_all = train['price'].valuesX_test_all = test[['size', 'bedrooms', 'age', 'distance']].valuesy_test_all = test['price'].valuesmean = X_train_all.mean(axis=0)std = X_train_all.std(axis=0)X_train_scaled = standardize(X_train_all, mean, std)X_test_scaled = standardize(X_test_all, mean, std)# Add interceptX_train_s = np.hstack([np.ones((X_train_scaled.shape[0], 1)), X_train_scaled])X_test_s = np.hstack([np.ones((X_test_scaled.shape[0], 1)), X_test_scaled])print('Scaled feature means:')print(X_train_scaled.mean(axis=0))print('Scaled feature stds:')print(X_train_scaled.std(axis=0))

## 7. Multiple Linear Regression from ScratchUse all four features. The math is the same, just with more columns.

In [ ]:
# Normal equation on scaled dataw_multi = np.linalg.pinv(X_train_s.T @ X_train_s) @ X_train_s.T @ y_train_all# Predicty_pred_multi = X_test_s @ w_multimse_multi = np.mean((y_test_all - y_pred_multi) ** 2)rmse_multi = np.sqrt(mse_multi)mae_multi = np.mean(np.abs(y_test_all - y_pred_multi))print('Multiple Regression Results:')print(f'w0 (intercept): {w_multi[0]:.2f}')print(f'w1 (size):      {w_multi[1]:.2f}')print(f'w2 (bedrooms):  {w_multi[2]:.2f}')print(f'w3 (age):       {w_multi[3]:.2f}')print(f'w4 (distance):  {w_multi[4]:.2f}')print(f'\nMSE:  {mse_multi:.2f}')print(f'RMSE: {rmse_multi:.2f}')print(f'MAE:  {mae_multi:.2f}')

## 8. Gradient Descent on Multiple FeaturesWith scaled features, the learning rate can be much larger.

In [ ]:
w_init_multi = np.zeros(X_train_s.shape[1])
alpha = 0.1
n_iter = 500
w_gd_multi, costs_multi = gradient_descent(
    X_train_s, y_train_all, w_init_multi, alpha, n_iter
)
print(f"GD final cost: {costs_multi[-1]:.2f}")
print("GD weights:")
print(w_gd_multi)
y_pred_gd = X_test_s @ w_gd_multi

rmse_gd = np.sqrt(np.mean((y_test_all - y_pred_gd) ** 2))
print(f"\nGD RMSE on test: {rmse_gd:.2f}")
plt.figure(figsize=(8, 4))
plt.plot(costs_multi)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.title("Gradient Descent Convergence (Multiple Features)")
plt.show()


## 9. Evaluation MetricsDifferent metrics tell different stories. R2 tells how much variance is explained.

In [ ]:
def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - (ss_res / ss_tot)


def adjusted_r2(y_true, y_pred, n, p):
    r2 = r2_score(y_true, y_pred)
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)


r2 = r2_score(y_test_all, y_pred_multi)
adj_r2 = adjusted_r2(y_test_all, y_pred_multi, len(y_test_all), X_test_s.shape[1] - 1)
print("Evaluation Metrics:")
print(f"MAE:           {mae_multi:.2f}")
print(f"MSE:           {mse_multi:.2f}")
print(f"RMSE:          {rmse_multi:.2f}")
print(f"R2:            {r2:.4f}")
print(f"Adjusted R2:   {adj_r2:.4f}")


## 10. Scikit-Learn ComparisonCompare our from-scratch implementation with scikit-learn. The results should match.

In [ ]:
from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_squared_error, mean_absolute_error

model = LinearRegression()
model.fit(X_train_scaled, y_train_all)
y_pred_sk = model.predict(X_test_scaled)
print("Scikit-Learn Results:")
print(f"Intercept: {model.intercept_:.2f}")
print(f"Coef:      {model.coef_}")
print(f"RMSE:      {np.sqrt(mean_squared_error(y_test_all, y_pred_sk)):.2f}")
print(f"MAE:       {mean_absolute_error(y_test_all, y_pred_sk):.2f}")
print("\nOur from-scratch intercept and coefs:")
print(f"Intercept: {w_multi[0]:.2f}")
print(f"Coef:      {w_multi[1:]}")


## 11. Regularization — Ridge from ScratchRidge adds a penalty for large weights: lambda * sum(w^2). This helps prevent overfitting.

In [ ]:
def ridge_normal_equation(X, y, lambda_):
    """Ridge regression closed-form solution."""
    m, n = X.shape
    I = np.eye(n)
    I[0, 0] = 0  # do not regularize the intercept
    return np.linalg.pinv(X.T @ X + lambda_ * I) @ X.T @ y


w_ridge = ridge_normal_equation(X_train_s, y_train_all, lambda_=1.0)
y_pred_ridge = X_test_s @ w_ridge

rmse_ridge = np.sqrt(np.mean((y_test_all - y_pred_ridge) ** 2))
print("Ridge Regression Results:")
print(f"Lambda: 1.0")
print(f"Weights: {w_ridge}")
print(f"RMSE:   {rmse_ridge:.2f}")  # Compare plain vs ridge coefficients

print("\nPlain vs Ridge Coefficients:")
comparison = pd.DataFrame(
    {
        "feature": ["size", "bedrooms", "age", "distance"],
        "plain": w_multi[1:],
        "ridge": w_ridge[1:],
    }
)
print(comparison)


## 12. Residual AnalysisResiduals are the differences between actual and predicted values. Good residuals should look random.

In [ ]:
residuals = y_test_all - y_pred_multiplt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.scatter(y_pred_multi, residuals, alpha=0.6)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted Price")
plt.ylabel("Residual")
plt.title("Residuals vs. Predicted")
plt.subplot(1, 2, 2)
plt.hist(residuals, bins=20, edgecolor="black")
plt.xlabel("Residual")
plt.ylabel("Count")
plt.title("Residual Distribution")
plt.tight_layout()
plt.show()
print(f"Residual mean: {residuals.mean():.2f}")
print(f"Residual std:  {residuals.std():.2f}")


## 13. Learning Rate ExperimentsTry different learning rates to see the effect on convergence.

In [ ]:
alphas = [0.01, 0.05, 0.1, 0.5]
colors = ["blue", "green", "red", "purple"]
plt.figure(figsize=(10, 5))
for alpha, color in zip(alphas, colors):
    w_init = np.zeros(X_train_s.shape[1])
    _, costs = gradient_descent(X_train_s, y_train_all, w_init, alpha, 200)
    plt.plot(costs, color=color, label=f"alpha={alpha}")
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.title("Effect of Learning Rate")
plt.legend()
plt.show()
print("Observations:")
print("- Too small alpha: slow convergence")
print("- Too large alpha: cost may increase or oscillate")
print("- Just right: smooth, fast decrease")


## 14. Exercises### Beginner1. Compute MSE, RMSE, MAE, and R2 for the simple regression. Compare with the multiple regression.2. Predict the price of a 2000 sq ft house using the simple model.3. Plot the cost history for different numbers of iterations.### Intermediate1. Implement Lasso regularization from scratch using coordinate descent or subgradient descent.2. Add polynomial features (e.g., size squared) and see if performance improves.3. Implement k-fold cross-validation manually.### Advanced1. Derive the gradient of the MSE cost with respect to each weight.2. Implement mini-batch gradient descent.3. Build a complete pipeline: generate data, train, validate, and test a final model.

## 15. Key TakeawaysAfter completing this project, you should understand:- The hypothesis function for linear regression- The MSE cost function and why it is used- How gradient descent updates weights- Why feature scaling matters- The closed-form normal equation- How regularization prevents overfitting- How to evaluate regression models- How to interpret residuals### Next Project**Project 5: Logistic Regression** — the same foundation, but for classification.